In [ ]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[2]))

# print(Path.cwd().resolve().parents[2])

In [ ]:
import json
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

In [ ]:

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_JSON  = "data.json"       # ← path to your JSON file
OUTPUT_XLSX = "output.xlsx"     # ← desired output file name

HEADER_BG   = "2F5597"         # dark-blue header background  (hex, no #)
HEADER_FG   = "FFFFFF"         # white header text
ODD_ROW_BG  = "DCE6F1"         # light-blue  for odd  rows
EVEN_ROW_BG = "FFFFFF"         # white        for even rows
FONT_NAME   = "Arial"
# ─────────────────────────────────────────────────────────────────────────────


def thin_border():
    side = Side(style="thin", color="B0B0B0")
    return Border(left=side, right=side, top=side, bottom=side)


def load_json(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Accept both a list-of-dicts and a single dict (wrap in list)
    if isinstance(data, dict):
        data = [data]
    return data


def write_excel(data: list[dict], output_path: str) -> None:
    df = pd.DataFrame(data)

    wb = Workbook()
    ws = wb.active
    ws.title = "Data"

    header_fill = PatternFill("solid", fgColor=HEADER_BG)
    header_font = Font(name=FONT_NAME, bold=True, color=HEADER_FG, size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left        = Alignment(horizontal="left",   vertical="center", wrap_text=True)

    # ── Write header row ──────────────────────────────────────────────────────
    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center
        cell.border    = thin_border()

    ws.row_dimensions[1].height = 30

    # ── Write data rows with alternating colours ──────────────────────────────
    odd_fill  = PatternFill("solid", fgColor=ODD_ROW_BG)
    even_fill = PatternFill("solid", fgColor=EVEN_ROW_BG)

    for row_idx, row in enumerate(df.itertuples(index=False), start=2):
        fill = odd_fill if row_idx % 2 != 0 else even_fill
        for col_idx, value in enumerate(row, start=1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.fill      = fill
            cell.font      = Font(name=FONT_NAME, size=10)
            cell.alignment = left
            cell.border    = thin_border()
        ws.row_dimensions[row_idx].height = 20

    # ── Auto-fit column widths ────────────────────────────────────────────────
    for col_idx, col_name in enumerate(df.columns, start=1):
        col_values  = [str(col_name)] + [str(v) for v in df.iloc[:, col_idx - 1]]
        max_len     = max(len(v) for v in col_values)
        ws.column_dimensions[ws.cell(1, col_idx).column_letter].width = min(max_len + 4, 50)

    # ── Freeze header row ─────────────────────────────────────────────────────
    ws.freeze_panes = "A2"

    wb.save(output_path)
    print(f"✅  Saved → {output_path}  ({len(df)} rows × {len(df.columns)} columns)")


if __name__ == "__main__":
    data = load_json(INPUT_JSON)
    write_excel(data, OUTPUT_XLSX)

/var/folders/50/3s07yszj1b331d4q48gw2vyr0000gq/T/ipykernel_59599/3012706575.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_details["DATE_OF_SERVICE"] = pd.to_datetime(df_details["DATE_OF_SERVICE"]).dt.strftime("%Y-%m-%d")


Saved → ../../../input-files/output/missing-invoice-billings.xlsx
